# Lineups Bronze Layer Ingestion

This notebook ingests StatsBomb lineup data from raw JSON files and transforms it into a bronze layer with the following schema:

## Bronze Schema
* **match_id** (BIGINT) - Extracted from filename
* **team_id** (INT) - Team identifier
* **team_name** (STRING) - Team name
* **player_id** (INT) - Player identifier
* **player_name** (STRING) - Full player name
* **player_nickname** (STRING) - Player nickname (nullable)
* **jersey_number** (INT) - Jersey number
* **country_id** (INT) - Country identifier
* **country_name** (STRING) - Country name
* **cards_json** (STRING) - JSON array of cards received (card_type, period, reason, time)
* **positions_json** (STRING) - JSON array of position changes during match
* **raw_json** (STRING) - Full original player JSON for schema flexibility
* **ingestion_ts** (TIMESTAMP) - Ingestion timestamp

## Data Volume
* **15,266 player records** across **416 matches**
* **39 teams**, **1,150 unique players**
* **86 countries** represented
* **1,374 player records** with cards

## Top Teams
* Liverpool (39 matches), Tottenham Hotspur (39), Chelsea (38)

## Top Countries
* England (4,092 players), Spain (1,707), France (1,006)

In [0]:
import sys
sys.path.append('/Workspace/Users/pawanvirat32@gmail.com/MatchPulse')

from config.paths import LINEUPS_RAW, LINEUPS_BRONZE

print(f"Reading lineups from: {LINEUPS_RAW}")
print(f"Will write to: {LINEUPS_BRONZE}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, TimestampType

# Read raw lineups JSON
df_raw = spark.read \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .json(LINEUPS_RAW)

# Add match_id from file path
df_with_match = df_raw.withColumn(
    "match_id",
    F.regexp_extract(F.col("_metadata.file_path"), r"/(\d+)\.json", 1).cast("bigint")
)

# Explode the lineup array to get one row per player
df_exploded = df_with_match \
    .withColumn("player", F.explode(F.col("lineup"))) \
    .select(
        F.col("match_id"),
        F.col("team_id"),
        F.col("team_name"),
        F.col("player.*")
    )

# Flatten to bronze schema
df_bronze = df_exploded.select(
    F.col("match_id").cast("bigint").alias("match_id"),
    F.col("team_id").cast("int").alias("team_id"),
    F.col("team_name").alias("team_name"),
    F.col("player_id").cast("int").alias("player_id"),
    F.col("player_name").alias("player_name"),
    F.col("player_nickname").alias("player_nickname"),
    F.col("jersey_number").cast("int").alias("jersey_number"),
    F.col("country.id").cast("int").alias("country_id"),
    F.col("country.name").alias("country_name"),
    F.to_json(F.col("cards")).alias("cards_json"),
    F.to_json(F.col("positions")).alias("positions_json"),
    F.to_json(F.struct(F.col("*"))).alias("raw_json"),
    F.current_timestamp().alias("ingestion_ts")
)

print(f"Total lineup records to ingest: {df_bronze.count():,}")
print("\nBronze schema:")
df_bronze.printSchema()
print("\nSample records:")
display(df_bronze.limit(10))

In [0]:
# Write the bronze data to S3
df_bronze.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(LINEUPS_BRONZE)

print(f"Successfully wrote lineups to bronze layer: {LINEUPS_BRONZE}")
print(f"Total records written: {df_bronze.count():,}")

In [0]:
# Read back from bronze to verify
df_verify = spark.read.parquet(LINEUPS_BRONZE)

print("Bronze layer statistics:")
print(f"Total lineup records: {df_verify.count():,}")
print(f"Unique matches: {df_verify.select('match_id').distinct().count():,}")
print(f"Unique teams: {df_verify.select('team_id').distinct().count():,}")
print(f"Unique players: {df_verify.select('player_id').distinct().count():,}")
print(f"Unique countries: {df_verify.select('country_id').distinct().count():,}")

print("\nTeam distribution (top 10):")
df_verify.groupBy('team_name') \
    .agg(F.countDistinct('match_id').alias('matches')) \
    .orderBy(F.desc('matches')) \
    .limit(10) \
    .show(truncate=False)

print("\nCountry distribution (top 10):")
df_verify.groupBy('country_name') \
    .count() \
    .orderBy(F.desc('count')) \
    .limit(10) \
    .show(truncate=False)

print("\nPlayers with cards:")
df_with_cards = df_verify.filter(F.col('cards_json') != '[]')
print(f"Total players with cards: {df_with_cards.count():,}")

print("\nSample bronze records:")
display(df_verify.limit(5))

In [0]:
display(df_raw.limit(20))

In [0]:
display(df_exploded.limit(25))